# LeetCode #1335: Minimum Difficulty of a Job Schedule

https://leetcode.com/problems/minimum-difficulty-of-a-job-schedule/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^d \cdot n)$ | $O(n \cdot d)$ |
| **Optimal: DP ★** | $O(n^2 \cdot d)$ | $O(n \cdot d)$ |

---

## Understanding the Methods

### Brute Force
Try every way to partition jobs into $d$ groups with at least one job per day, computing the daily max for each partition. Exponential partitions make this infeasible.

### Optimal: DP ★
`dp[day][i]` = minimum schedule difficulty for the first $i$ jobs in `day` days. For each day, try every possible last job $j$ in that day; the cost for day is $\max(\text{jobDifficulty}[j..i-1])$ and the total is `dp[day-1][j] + max`. The inner max is computed in $O(n)$ per (day, i) pair, giving $O(n^2 \cdot d)$ overall.

**Constraints:**
* $1 \leq \text{jobDifficulty.length} \leq 300$
* $1 \leq d \leq 10$
* $0 \leq \text{jobDifficulty}[i] \leq 1000$


## Solutions

### C#

In [ ]:
public class Solution {
    public int MinDifficulty(int[] jobDifficulty, int d) {
        int n = jobDifficulty.Length;
        if (n < d) return -1; // Cannot schedule fewer jobs than days

        const int INF = int.MaxValue / 2;
        // dp[i] = min difficulty to schedule the first i jobs in the current number of days
        int[] dp = new int[n + 1];
        int[] prev = new int[n + 1];

        // Base: day 1 — best we can do is take all jobs 1..i, cost = max(0..i-1)
        for (int i = 0; i <= n; i++) prev[i] = INF;
        prev[0] = 0;
        int runMax = 0;
        for (int i = 1; i <= n; i++) {
            runMax = Math.Max(runMax, jobDifficulty[i - 1]);
            prev[i] = runMax; // Day 1 must take all jobs up to i
        }

        for (int day = 2; day <= d; day++) {
            for (int i = 0; i <= n; i++) dp[i] = INF;

            for (int i = day; i <= n; i++) {
                // Try all possible last-job boundaries j for today's batch
                int dayMax = 0;
                for (int j = i; j >= day; j--) {
                    // Extend today's batch leftward, updating the daily maximum
                    dayMax = Math.Max(dayMax, jobDifficulty[j - 1]);
                    if (prev[j - 1] < INF)
                        dp[i] = Math.Min(dp[i], prev[j - 1] + dayMax);
                }
            }

            // Roll over to the next day
            int[] temp = prev; prev = dp; dp = temp;
        }

        return prev[n] >= INF ? -1 : prev[n];
    }
}

### Python

In [ ]:
class Solution:
    def minDifficulty(self, job_difficulty: list[int], d: int) -> int:
        n = len(job_difficulty)
        if n < d:
            return -1  # Cannot schedule fewer jobs than days

        INF = float('inf')
        # prev[i] = min difficulty to schedule the first i jobs in the current number of days
        prev = [INF] * (n + 1)
        prev[0] = 0

        # Base: day 1 — best we can do is take all jobs 1..i, cost = max(0..i-1)
        run_max = 0
        for i in range(1, n + 1):
            run_max = max(run_max, job_difficulty[i - 1])
            prev[i] = run_max

        for day in range(2, d + 1):
            dp = [INF] * (n + 1)

            for i in range(day, n + 1):
                # Try all possible last-job boundaries j for today's batch
                day_max = 0
                for j in range(i, day - 1, -1):
                    # Extend today's batch leftward, updating the daily maximum
                    day_max = max(day_max, job_difficulty[j - 1])
                    if prev[j - 1] < INF:
                        dp[i] = min(dp[i], prev[j - 1] + day_max)

            prev = dp

        return -1 if prev[n] == INF else prev[n]


### Go

In [ ]:
func minDifficulty(jobDifficulty []int, d int) int {
    n := len(jobDifficulty)
    if n < d {
        return -1 // Cannot schedule fewer jobs than days
    }

    const INF = 1<<30
    // prev[i] = min difficulty to schedule the first i jobs in the current number of days
    prev := make([]int, n+1)
    for i := range prev { prev[i] = INF }
    prev[0] = 0

    // Base: day 1
    runMax := 0
    for i := 1; i <= n; i++ {
        if jobDifficulty[i-1] > runMax { runMax = jobDifficulty[i-1] }
        prev[i] = runMax
    }

    dp := make([]int, n+1)
    for day := 2; day <= d; day++ {
        for i := range dp { dp[i] = INF }

        for i := day; i <= n; i++ {
            // Try all possible last-job boundaries j for today's batch
            dayMax := 0
            for j := i; j >= day; j-- {
                // Extend today's batch leftward, updating the daily maximum
                if jobDifficulty[j-1] > dayMax { dayMax = jobDifficulty[j-1] }
                if prev[j-1] < INF {
                    if prev[j-1]+dayMax < dp[i] { dp[i] = prev[j-1] + dayMax }
                }
            }
        }
        prev, dp = dp, prev
    }

    if prev[n] >= INF { return -1 }
    return prev[n]
}

### Rust

In [ ]:
impl Solution {
    pub fn min_difficulty(job_difficulty: Vec<i32>, d: i32) -> i32 {
        let n = job_difficulty.len();
        let d = d as usize;
        if n < d { return -1; } // Cannot schedule fewer jobs than days

        const INF: i32 = i32::MAX / 2;
        // prev[i] = min difficulty to schedule the first i jobs in the current number of days
        let mut prev = vec![INF; n + 1];
        prev[0] = 0;

        // Base: day 1
        let mut run_max = 0;
        for i in 1..=n {
            run_max = run_max.max(job_difficulty[i - 1]);
            prev[i] = run_max;
        }

        for day in 2..=d {
            let mut dp = vec![INF; n + 1];

            for i in day..=n {
                // Try all possible last-job boundaries j for today's batch
                let mut day_max = 0;
                for j in (day..=i).rev() {
                    // Extend today's batch leftward, updating the daily maximum
                    day_max = day_max.max(job_difficulty[j - 1]);
                    if prev[j - 1] < INF {
                        dp[i] = dp[i].min(prev[j - 1] + day_max);
                    }
                }
            }
            prev = dp;
        }

        if prev[n] >= INF { -1 } else { prev[n] }
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `jobDifficulty = [6, 5, 4, 3, 2, 1], d = 2`
Best split: day 1 does job 0 (difficulty 6), day 2 does jobs 1-5 (max = 5). Total = 11. The DP tries all splits and confirms 11.

### 2. Slightly Complex
**Input:** `jobDifficulty = [9, 9, 9], d = 4`
Three jobs, four days: impossible since at least one job per day is required. Returns $-1$ immediately from the early guard.

### 3. Edge Case: Time Factor
**Input:** `n = 300, d = 10`, all values 1000
All $O(n^2 \cdot d) = 900{,}000$ DP cells are filled — the worst-case path. Each day's cost is 1000 regardless of split, so total = 10000.

### 4. Edge Case: Space Factor
**Input:** `n = 10, d = 10` (one job per day)
Only one valid schedule. The DP array has length 11, and only one element per row is reachable — minimising both time and space in practice.

### 5. Almost-Impossible but Plausible
**Input:** `jobDifficulty = [1000, 0, 0, \ldots, 0, 1000]` ($n = 300$, $d = 10$)
Placing the two spikes on separate days costs $1000 \times 2$ plus minimal 0s on other days. The DP correctly isolates each spike, demonstrating that the inner max-scan finds the optimal boundary even when surrounded by zeros.
